In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

print(os.getenv("LANGSMITH_TRACING"))
print(os.getenv("LANGSMITH_ENDPOINT"))
print(os.getenv("LANGSMITH_PROJECT"))
print(os.getenv("LANGSMITH_API_KEY")[:10] + "...")

true
https://api.smith.langchain.com
RAG_Pipeline
lsv2_pt_5c...


In [ ]:
import pandas as pd
from langsmith import Client

# =====================================================
# Configuration
# =====================================================

EXCEL_FILE = "ABB_RAG_Dataset.xlsx"      # Path to your Excel file
SHEET_NAME = "Sheet1"

DATASET_NAME = "Calculator Dataset2"

# =====================================================
# Read Excel
# =====================================================

df = pd.read_excel(
    EXCEL_FILE,
    sheet_name=SHEET_NAME
)

print(df.head())

# =====================================================
# Create LangSmith Client
# =====================================================

client = Client()

# =====================================================
# Create or Read Dataset
# =====================================================

try:

    dataset = client.create_dataset(
        dataset_name=DATASET_NAME,
        description="Dataset imported from Excel"
    )

    print("Dataset created.")

except Exception:

    dataset = client.read_dataset(
        dataset_name=DATASET_NAME
    )

    print("Dataset already exists.")

# =====================================================
# Convert Excel rows
# =====================================================

inputs = []
outputs = []

for _, row in df.iterrows():

    inputs.append(
        {
            "question": str(row["inputs"])
        }
    )

    outputs.append(
        {
            "answer": str(row["reference_outputs"])
        }
    )

# =====================================================
# Upload Examples
# =====================================================

client.create_examples(
    dataset_id=dataset.id,
    inputs=inputs,
    outputs=outputs,
)

print(f"{len(inputs)} examples uploaded successfully.")

In [6]:
import logging
from pathlib import Path

LOG_DIR = Path("logs")
LOG_DIR.mkdir(exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(message)s",
    handlers=[
        logging.FileHandler(LOG_DIR / "rag_pipeline.log", encoding="utf-8"),
        logging.StreamHandler()
    ]
)

logger = logging.getLogger("RAG")

In [2]:
from sentence_transformers import SentenceTransformer, CrossEncoder
from pgvector.psycopg2 import register_vector
import psycopg2
import ollama
from dotenv import load_dotenv
load_dotenv()

from langsmith import traceable, get_current_run_tree
HOST = "localhost"
PORT = 5434
USER = "admin"
PASSWORD = "pass123"
DATABASE = "RAG_POC"

TABLE = "rag_chunks"

EMBED_MODEL = "BAAI/bge-m3"
RERANK_MODEL = "BAAI/bge-reranker-v2-m3"
OLLAMA_MODEL = "qwen3:8b"
print("Loading models...")

embed_model = SentenceTransformer(EMBED_MODEL)

reranker = CrossEncoder(RERANK_MODEL)
@traceable(name="RAG Retrieval", run_type="chain")
def data_retrival(
    SEARCH_QUERY,
    QUESTION,
    RULES,
    TOP_K_RETRIEVAL=10,
    TOP_K_FINAL=5,
):

    # ==========================================================
    # Embedding
    # ==========================================================

    @traceable(name="Embedding", run_type="embedding")
    def create_embedding(text):
        return embed_model.encode(
            text,
            normalize_embeddings=True
        ).tolist()

    query_embedding = create_embedding(SEARCH_QUERY)
    embedding_str = "[" + ",".join(map(str, query_embedding)) + "]"

    # ==========================================================
    # Vector Search
    # ==========================================================

    @traceable(name="Vector Search", run_type="retriever")
    def semantic_search(embedding, top_k):

        conn = psycopg2.connect(
            host=HOST,
            port=PORT,
            user=USER,
            password=PASSWORD,
            dbname=DATABASE,
        )

        register_vector(conn)

        cur = conn.cursor()

        sql = f"""
        SELECT
            page_content,
            source,
            metadata,
            embedding <=> %s::vector AS distance
        FROM {TABLE}
        ORDER BY distance
        LIMIT %s;
        """

        cur.execute(sql, (embedding, top_k))

        rows = cur.fetchall()

        cur.close()
        conn.close()

        return rows

    rows = semantic_search(
        embedding_str,
        TOP_K_RETRIEVAL
    )

    if not rows:
        logger.info("No documents found.")
        return "No documents found."

    logger.info("=" * 80)
    logger.info("TOP VECTOR SEARCH RESULTS")
    logger.info("=" * 80)

    for i, row in enumerate(rows, start=1):

        page_content, source, metadata, distance = row

        logger.info(f"Rank {i}")
        logger.info(f"Distance : {distance:.5f}")
        logger.info(page_content[:300])

    # ==========================================================
    # Reranker
    # ==========================================================

    @traceable(name="CrossEncoder Reranker", run_type="chain")
    def rerank_documents(query, rows):

        rerank_query = f"""
Search Query:
{SEARCH_QUERY}

User Question:
{QUESTION}
"""

        pairs = [
            (rerank_query, row[0])
            for row in rows
        ]

        scores = reranker.predict(pairs)

        reranked = []

        for row, score in zip(rows, scores):

            page_content, source, metadata, distance = row

            reranked.append(
                {
                    "page_content": page_content,
                    "source": source,
                    "metadata": metadata,
                    "distance": distance,
                    "rerank_score": float(score),
                }
            )

        reranked.sort(
            key=lambda x: x["rerank_score"],
            reverse=True,
        )

        return reranked[:TOP_K_FINAL]

    reranked = rerank_documents(
        SEARCH_QUERY,
        rows,
    )

    logger.info("=" * 80)
    logger.info("TOP RERANKED CHUNKS")
    logger.info("=" * 80)

    for i, doc in enumerate(reranked, start=1):

        logger.info(f"Rank {i}")
        logger.info(f"Distance : {doc['distance']:.5f}")
        logger.info(f"Reranker : {doc['rerank_score']:.5f}")

    # ==========================================================
    # Prompt Builder
    # ==========================================================

    @traceable(name="Prompt Builder", run_type="prompt")
    def build_prompt(context, question, rules):

        STANDARD_RULES = [
            "Use ONLY the supplied context.",
            "Do NOT use outside knowledge.",
            "Return only the requested information.",
            "Do not explain unless requested.",
            "Do not summarize unless requested.",
            "If the answer appears explicitly in the context, return it exactly.",
            "If the context partially answers the question, return the available information.",
            "Only reply 'Data is not available.' when no relevant information exists.",
        ]

        all_rules = STANDARD_RULES + rules

        rule_text = "\n".join(
            f"{i}. {r}"
            for i, r in enumerate(all_rules, 1)
        )

        prompt = f"""
You are a RAG assistant.

Answer ONLY from the supplied context.

Context:
{context}

Question:
{question}

Rules:
{rule_text}

Answer:
"""

        return prompt

    context = "\n\n".join(
        doc["page_content"]
        for doc in reranked
    )

    prompt = build_prompt(
        context,
        QUESTION,
        RULES,
    )

    # ==========================================================
    # LLM
    # ==========================================================

    @traceable(name="Answer Generation", run_type="llm")
    def generate_answer(prompt):

        return ollama.chat(
            model=OLLAMA_MODEL,
            messages=[
                {
                    "role": "user",
                    "content": prompt,
                }
            ],
        )

    response = generate_answer(prompt)

    answer = response["message"]["content"]

    if "</think>" in answer:
        answer = answer.split("</think>", 1)[1].strip()

    logger.info("=" * 80)
    logger.info("FINAL ANSWER")
    logger.info("=" * 80)
    logger.info(answer)

    logger.info("=" * 80)
    logger.info("RETRIEVAL STATISTICS")
    logger.info("=" * 80)
    logger.info(f"Vector Search Top-K : {TOP_K_RETRIEVAL}")
    logger.info(f"Reranker Top-K      : {TOP_K_FINAL}")
    logger.info(f"Context Chunks      : {len(reranked)}")
    logger.info(f"Prompt Characters   : {len(prompt)}")

    with open("../console.md", "w", encoding="utf-8") as f:
        f.write(answer)

    return {
    "answer": answer,
    "context": context,
    "search_query": SEARCH_QUERY,
    "rules": RULES,
    "retrieved_chunks": [
        doc["page_content"] for doc in reranked
    ],
    }

c:\RAG_POC\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading models...


Loading weights: 100%|██████████| 393/393 [00:00<00:00, 10306.77it/s]


In [3]:
import json
import ollama

@traceable(name="Question Planner", run_type="llm"
)
def prepare_question(question):
    """
    Generate:
        1. Optimized semantic search query
        2. Question-specific answer rules
    """

    planner_prompt = f"""
You are an expert Query Planner for a Retrieval-Augmented Generation (RAG) system.

Your job is NOT to answer the user's question.

Your ONLY job is to prepare retrieval instructions.

=====================================================================

USER QUESTION

{question}

=====================================================================

TASK 1

Generate the BEST semantic search query.

Guidelines:

- Preserve technical terminology exactly.
- Preserve product names.
- Preserve menu names.
- Preserve UI labels.
- Preserve figure names.
- Preserve table names.
- Preserve chapter names.
- Preserve command names.
- Preserve important noun phrases.
- Remove conversational words.
- Remove answer formatting instructions.
- Keep the search query concise.
- Maximum 12 words.

=====================================================================

TASK 2

Generate ONLY question-specific answer rules.

Examples of GOOD rules:


- Preserve the original numbering.
- Return the complete procedure.
- Preserve the original wording.

DO NOT generate:

- Generic RAG rules.
- Explanations.
- New questions.
- Configuration steps.
- Hallucinated information.

=====================================================================

Return ONLY valid JSON.

Expected JSON format:

{{
    "search_query": "...",
    "rules": [
        "...",
        "...",
        "..."
    ]
}}

Example

User Question:

Give me the table of Hot Keys in the Operator Workplace.
Print the output in table format.
Skip first 7 rows.

Expected Output:

{{
    "search_query": "Hot Keys, Operator Workplace",

    "rules": [
        "Return the answer in table format.",
        "Skip the first 7 rows.",
        "Preserve the original row order."
    ]
}}

IMPORTANT

Return ONLY JSON.

Do NOT return Markdown.

Do NOT wrap the JSON inside ```json.

Do NOT explain your reasoning.

Do NOT include <think>.
"""

    response = ollama.chat(
        model=OLLAMA_MODEL,
        messages=[
            {
                "role": "user",
                "content": planner_prompt
            }
        ]
    )

    text = response["message"]["content"].strip()

    # ----------------------------------------------------------
    # Remove <think>...</think> (Qwen3 sometimes generates this)
    # ----------------------------------------------------------

    if "</think>" in text:
        text = text.split("</think>", 1)[1].strip()

    # ----------------------------------------------------------
    # Remove Markdown code fences
    # ----------------------------------------------------------

    if text.startswith("```"):
        text = text.replace("```json", "")
        text = text.replace("```", "")
        text = text.strip()

    # ----------------------------------------------------------
    # Parse JSON
    # ----------------------------------------------------------

    try:
        result = json.loads(text)

    except json.JSONDecodeError:

        logger.info("=" * 80)
        logger.info("INVALID JSON RETURNED BY PLANNER")
        logger.info("=" * 80)
        logger.info(text)

        raise

    # ----------------------------------------------------------
    # Print Planner Output
    # ----------------------------------------------------------

    logger.info("\n" + "=" * 80)
    logger.info("SEARCH QUERY")
    logger.info("=" * 80)
    logger.info(result["search_query"])

    logger.info("\n" + "=" * 80)
    logger.info("QUESTION SPECIFIC RULES")
    logger.info("=" * 80)

    for rule in result["rules"]:
        logger.info(f"- {rule}")

    logger.info("-------")

    return result["search_query"], result["rules"]

In [7]:
from dotenv import load_dotenv
load_dotenv()

import json
import ollama

from langsmith import traceable
from langsmith.evaluation import evaluate

# ==========================================================
# RAG Pipeline
# ==========================================================

@traceable(name="ABB RAG Pipeline", run_type="chain")
def rag_pipeline(inputs):

    question = inputs["question"]

    print("=" * 80)
    print("Question:", question)

    search_query, rules = prepare_question(question)

    result = data_retrival(
        SEARCH_QUERY=search_query,
        QUESTION=question,
        RULES=rules,
    )

    return {
        "answer": result["answer"],
        "context": result["context"],
        "search_query": result["search_query"],
        "rules": result["rules"],
        "retrieved_chunks": result["retrieved_chunks"],
    }


# ==========================================================
# Generic Judge
# ==========================================================

@traceable(name="Generic Judge", run_type="llm")
def _judge(metric_name, prompt):

    response = ollama.chat(
        model=OLLAMA_MODEL,
        messages=[
            {
                "role": "user",
                "content": prompt,
            }
        ],
    )

    text = response["message"]["content"].strip()

    if "</think>" in text:
        text = text.split("</think>", 1)[1].strip()

    if text.startswith("```"):
        text = (
            text.replace("```json", "")
            .replace("```", "")
            .strip()
        )

    try:
        result = json.loads(text)

        return {
            "key": metric_name,
            "score": float(result["score"]),
            "comment": result["reason"],
        }

    except Exception:

        return {
            "key": metric_name,
            "score": 0.0,
            "comment": text,
        }


# ==========================================================
# Overall LLM Judge
# ==========================================================

@traceable(name="Overall LLM Judge", run_type="llm")
def llm_judge(outputs, reference_outputs, inputs):

    prompt = f"""
You are an expert RAG evaluator.

Question:
{inputs["question"]}

Expected Answer:
{reference_outputs["answer"]}

Generated Answer:
{outputs["answer"]}

Evaluate:

1. Correctness
2. Completeness
3. Relevance
4. Clarity

Return ONLY JSON.

{{
    "score":1,
    "reason":"..."
}}
"""

    return _judge("Overall LLM", prompt)


# ==========================================================
# Correctness
# ==========================================================

@traceable(name="Correctness Judge", run_type="llm")
def correctness_judge(outputs, reference_outputs, inputs):

    prompt = f"""
Question:
{inputs["question"]}

Expected Answer:
{reference_outputs["answer"]}

Generated Answer:
{outputs["answer"]}

Is the generated answer correct?

Score:

1 = Correct
0 = Incorrect

Return ONLY JSON.

{{
    "score":1,
    "reason":"..."
}}
"""

    return _judge("Correctness", prompt)


# ==========================================================
# Faithfulness
# ==========================================================

@traceable(name="Faithfulness Judge", run_type="llm")
def faithfulness_judge(outputs, reference_outputs, inputs):

    prompt = f"""
Question:
{inputs["question"]}

Retrieved Context:

{outputs["context"]}

Generated Answer:

{outputs["answer"]}

Determine whether every statement in the generated answer is supported by the retrieved context.

Ignore the reference answer.

Score:

1 = Fully Supported

0 = Unsupported Statements

Return ONLY JSON.

{{
    "score":1,
    "reason":"..."
}}
"""

    return _judge("Faithfulness", prompt)


# ==========================================================
# Hallucination
# ==========================================================

@traceable(name="Hallucination Judge", run_type="llm")
def hallucination_judge(outputs, reference_outputs, inputs):

    prompt = f"""
Question:
{inputs["question"]}

Retrieved Context:

{outputs["context"]}

Generated Answer:

{outputs["answer"]}

Does the answer contain any information not present in the retrieved context?

Score:

1 = No Hallucination

0 = Hallucinated

Return ONLY JSON.

{{
    "score":1,
    "reason":"..."
}}
"""

    return _judge("Hallucination", prompt)


# ==========================================================
# Answer Relevance
# ==========================================================

@traceable(name="Answer Relevance Judge", run_type="llm")
def answer_relevance_judge(outputs, reference_outputs, inputs):

    prompt = f"""
Question:

{inputs["question"]}

Generated Answer:

{outputs["answer"]}

Does the answer directly answer the user's question?

Score:

1 = Relevant

0 = Not Relevant

Return ONLY JSON.

{{
    "score":1,
    "reason":"..."
}}
"""

    return _judge("Answer Relevance", prompt)


# ==========================================================
# Retrieval Relevance
# ==========================================================

@traceable(name="Retrieval Relevance Judge", run_type="llm")
def retrieval_relevance_judge(outputs, reference_outputs, inputs):

    docs = "\n\n".join(outputs["retrieved_chunks"])

    prompt = f"""
Question:

{inputs["question"]}

Retrieved Documents:

{docs}

Determine whether these retrieved documents are relevant for answering the question.

Ignore the generated answer.

Score:

1 = Relevant

0 = Irrelevant

Return ONLY JSON.

{{
    "score":1,
    "reason":"..."
}}
"""

    return _judge("Retrieval Relevance", prompt)


# ==========================================================
# Run Evaluation
# ==========================================================

experiment = evaluate(
    rag_pipeline,
    data="Calculator Dataset2",
    evaluators=[
        llm_judge,
        correctness_judge,
        faithfulness_judge,
        hallucination_judge,
        answer_relevance_judge,
        retrieval_relevance_judge,
    ],
    experiment_prefix="ABB-RAG-v4",
)

print("\nExperiment Finished")
print(experiment)

View the evaluation results for experiment: 'ABB-RAG-v4-37c36d5b' at:
https://smith.langchain.com/o/fa3d1324-37d3-4d19-bf7e-c4846aca8587/datasets/7dfae4b8-23d1-4c40-8a14-aa2c60cb421c/compare?selectedSessions=35e80fdc-ae7c-4a38-a6bb-4ce38f7f0624




0it [00:00, ?it/s]

Question: which steps shoud be performed to open the workplace for the first time


2026-07-29 17:16:25,217 | INFO     | HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2026-07-29 17:16:25,218 | INFO     | 
2026-07-29 17:16:25,219 | INFO     | SEARCH QUERY
2026-07-29 17:16:25,219 | INFO     | ================================================================================
2026-07-29 17:16:25,220 | INFO     | Steps to open Workplace for first time
2026-07-29 17:16:25,220 | INFO     | 
2026-07-29 17:16:25,220 | INFO     | QUESTION SPECIFIC RULES
2026-07-29 17:16:25,221 | INFO     | ================================================================================
2026-07-29 17:16:25,221 | INFO     | - Preserve the original numbering.
2026-07-29 17:16:25,221 | INFO     | - Return the complete procedure.
2026-07-29 17:16:25,222 | INFO     | - Preserve the original wording.
2026-07-29 17:16:25,222 | INFO     | -------
Batches: 100%|██████████| 1/1 [00:00<00:00,  1.18it/s]
2026-07-29 17:16:26,200 | INFO     | ==============================================

Question: which protocal should be followed for Transfer of Responsibility


2026-07-29 17:17:48,074 | INFO     | HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2026-07-29 17:17:48,075 | INFO     | 
2026-07-29 17:17:48,076 | INFO     | SEARCH QUERY
2026-07-29 17:17:48,076 | INFO     | ================================================================================
2026-07-29 17:17:48,077 | INFO     | Transfer of Responsibility Protocol
2026-07-29 17:17:48,077 | INFO     | 
2026-07-29 17:17:48,077 | INFO     | QUESTION SPECIFIC RULES
2026-07-29 17:17:48,078 | INFO     | ================================================================================
2026-07-29 17:17:48,078 | INFO     | - Preserve the original numbering.
2026-07-29 17:17:48,078 | INFO     | - Return the complete procedure.
2026-07-29 17:17:48,078 | INFO     | - Preserve the original wording.
2026-07-29 17:17:48,078 | INFO     | -------
Batches: 100%|██████████| 1/1 [00:00<00:00, 10.40it/s]
2026-07-29 17:17:48,212 | INFO     | =================================================

Question: which figure number should i refer application bar configure for Alarm Logger Manager


2026-07-29 17:20:32,185 | INFO     | HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2026-07-29 17:20:32,186 | INFO     | 
2026-07-29 17:20:32,187 | INFO     | SEARCH QUERY
2026-07-29 17:20:32,188 | INFO     | ================================================================================
2026-07-29 17:20:32,188 | INFO     | Figure number for configuring Alarm Logger Manager in Application Bar
2026-07-29 17:20:32,189 | INFO     | 
2026-07-29 17:20:32,189 | INFO     | QUESTION SPECIFIC RULES
2026-07-29 17:20:32,190 | INFO     | ================================================================================
2026-07-29 17:20:32,190 | INFO     | - Preserve the original numbering.
2026-07-29 17:20:32,191 | INFO     | - Return the complete figure reference.
2026-07-29 17:20:32,191 | INFO     | - Do not include additional explanations.
2026-07-29 17:20:32,192 | INFO     | -------
Batches: 100%|██████████| 1/1 [00:00<00:00, 12.23it/s]
2026-07-29 17:20:32,321 | INFO     |


Experiment Finished
<ExperimentResults ABB-RAG-v4-37c36d5b>
